# W07A · A · 샘플링 원리와 실제 기록 분석
2026-10-12 · 주교재5.4–5.7 · CPU · 패키지0.1.11

작은 계산을 먼저 풀고 실제 SD 실행에서 저장한 텐서를 읽습니다. 이 노트북은 SD 가중치를 내려받거나 새 이미지를 생성하지 않습니다. 이전 실습·API 키·개인 파일 없이 독립적으로 실행합니다. 문제를 건너뛰어도 나머지 기본 셀은 실행됩니다.

[강의노트](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/course/notion/w07a/w07a-sampling.md) · [API 입력·출력과 정의](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/src/W07A-API.md) · [실험 기록지](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/course/notion/w07a/w07a-workbook.md)


In [ ]:
import sys, subprocess, os, time
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
PUBLIC_REF='2026-fall-w07a'
try:
    ready=all(version(p)==v for p,v in [('luna-genai','0.1.11'),('diffusers','0.40.0'),('transformers','5.15.1'),('gradio','6.26.0')])
except PackageNotFoundError: ready=False
if not ready:
    if any(p in sys.modules for p in ('luna_genai','diffusers','transformers','gradio')):
        raise RuntimeError('이전 모듈이 남아 있습니다. 런타임 재시작 후 첫 셀부터 실행하세요.')
    package=str(Path('src').resolve()) if Path('src/pyproject.toml').exists() else f'git+https://github.com/lunalab-ai/genAI.git@{PUBLIC_REF}#subdirectory=src'
    subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
torch.set_num_threads(2)
IN_COLAB='COLAB_RELEASE_TAG' in os.environ
print('Python',sys.version.split()[0],'luna-genai',version('luna-genai'),'CUDA',torch.cuda.is_available())


## 1. 읽을 기록과 실행 경계
[`load_sampling_records()`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/src/luna_genai/sampling.py#L188)는 패키지의 실제 T4 기록과 이미지·배열 해시를 검사합니다. 입력이 없고 네트워크·모델 실행도 없습니다. 반환 사전의 `runs`는 설정별 기록, `parity`는 수동 루프와 기본 파이프라인의 실측 비교입니다. `ASSETS`는 설치 패키지 안의 공개 수업 자산 경로입니다.


In [ ]:
from luna_genai.sampling import load_sampling_records, ASSETS, ddim_scalar
from luna_genai.stable_diffusion import cfg_combine
record=load_sampling_records()
display(pd.DataFrame(record['runs'])[['key','negative_prompt','guidance_scale','steps','device','seconds']])
print('실행 모드: 저장된 실제 T4 결과 분석 / 지금 SD 생성 아님')
print('기록 출처:',record['created_at'],record['environment'])
print('직접 루프 대 pipeline:',record['parity']['max_abs'])


### A1 · 무엇이 바뀌는가

“20번 생성 단계를 거치므로 U-Net을 20번 학습한다”를 고치세요. 고정되는 것과 바뀌는 것을 적으세요.
힌트: optimizer와 backward가 있는지 봅니다. 자가 점검: 반복과 학습을 구별했는지 확인합니다.


## 2. CFG: 그림 이전에 두 수부터
[`cfg_combine(u,c,s)`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/src/luna_genai/stable_diffusion.py#L17)는 같은 shape/dtype/device의 유한 torch.Tensor 두 개와 scale을 받습니다. 반환값은 같은 크기의 잡음 예측이며 가중치나 난수 상태를 바꾸지 않습니다. 예를 들어 u=(.2,-.1), c=(.5,.1), s=2이면 (.8,.3)입니다. API guidance≤1 분기와 수학식의 s=0은 구별합니다.


In [ ]:
u=torch.tensor([.2,-.1]); c=torch.tensor([.5,.1])
print('차이 c-u:',c-u)
print('함께 푸는 s=2:',cfg_combine(u,c,2.))


### A2 · CFG 빈칸과 수계산

s=3일 때 `guided_student = u + s * (____)`를 완성하고 각 성분 계산을 쓰세요.
힌트: 괄호는 조건 차이입니다. 자가 점검: s=1에서 c가 나오는 식인지 봅니다.


### A3 · negative 기준 바꾸기

n=(0,-.2), c=(.5,.1), s=2에서 n+s(c-n)을 계산하세요. 값이 달라졌다는 사실이 색 제거를 보장하는지도 설명하세요.
힌트: 결과는 여전히 잡음 예측입니다.


## 3. DDIM 한 단계의 계산
[`ddim_scalar(x_t, epsilon, alpha_t, alpha_previous)`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/src/luna_genai/sampling.py#L17)는 설명용 스칼라 계산입니다. alpha는 누적 계수이며 0<현재≤다음≤1입니다. 반환 사전은 predicted_x0와 previous 두 값을 담습니다. 모델을 실행하지 않으며 실제 scheduler를 대체하는 일반 알고리즘도 아닙니다.


In [ ]:
small=ddim_scalar(.6,.2,.25,.64)
print(small)
print('단순 뺄셈과 비교:',.6-.2)
assert abs(small['previous']-.8028718707889796)<1e-10


## 4. 실제 저장 latent를 읽고 shape와 상태를 확인
`np.load(..., allow_pickle=False)`로 숫자 배열만 읽습니다. 각 frame은 갱신 후의 latent (1,4,64,64)입니다. `input_t`는 그 갱신에 사용한 시점으로, 반복 횟수와 다른 값입니다. RMS는 배열의 크기 통계이며 화질 점수가 아닙니다.


In [ ]:
with np.load(ASSETS/'sampling-tensors.npz',allow_pickle=False) as archive:
    tensors={k:archive[k].copy() for k in archive.files}
base=record['runs'][0]
trace=pd.DataFrame(base['trace']);display(trace)
print('frame0:',tensors['base_frame_0'].shape,'final:',tensors['base_final'].shape)
fig,ax=plt.subplots(figsize=(8,3));ax.plot(trace['step'],trace['latent_rms_after'],'o-')
ax.set(xlabel='step',ylabel='latent RMS',title='Measured latent magnitude (NOT image quality)');ax.grid(alpha=.2);plt.show()


### A4 · shape와 비교 대상

(2,4,64,64)를 chunk(2)한 두 결과와 CFG 결합 후의 크기를 쓰세요. 두 갈래의 latent를 다르게 뽑으면 왜 비교가 달라질까요?
힌트: batch축만 나눕니다.


## 5. seed 연결 실험
아래는 모델과 무관한 작은 난수 실험입니다. `torch.Generator().manual_seed`의 반환값을 randn의 generator에 전달합니다. 호출될 때 Generator의 상태가 소비된다는 점을 확인합니다.


In [ ]:
g=torch.Generator().manual_seed(42)
first=torch.randn(4,generator=g); second=torch.randn(4,generator=g)
print('같은 객체, 연속 호출:',first,second)
print('같은가?',torch.equal(first,second))


### A5 · seed 디버깅

`seed=42; z=torch.randn(1,4,64,64)`를 고치고 새 Generator 두 개의 첫 출력이 같은지 검사하세요. 위 g의 연속 호출이 다른 이유도 쓰세요.
힌트: seed 변수 선언과 난수 호출을 연결합니다.


## 6. 최종 결과를 직접 보고 한계 기록
같은 seed의 base와 negative-red를 나란히 봅니다. 붉은색이 남아 있는지, 컵의 형태와 배경이 어떻게 달라졌는지 관찰합니다. 단일 seed의 관찰은 보편적인 효과 추정이 아닙니다.


In [ ]:
fig,axes=plt.subplots(1,4,figsize=(14,4))
for ax,r in zip(axes,record['runs']):
    ax.imshow(Image.open(ASSETS/r['file']));ax.set_title(r['key']);ax.axis('off')
plt.tight_layout();plt.show()


## 7. 마지막 웹 앱: 기록을 선택해 단계별 과정 확인
[`sampling_record_view(key)`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/src/luna_genai/sampling_app.py#L20)는 이미지·과정·설정·JSON 경로 네 값을 반환합니다. [`build_sampling_app()`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w07a/src/luna_genai/sampling_app.py#L49)은 모델 로드 없이 앱을 구성합니다. Colab에서는 마지막 공유 링크를 새 탭에서 열고 “기록과 중간 과정 보기”를 누르세요. 기록을 바꾸면 설정도 함께 바뀌어야 합니다. CPU 기록 모드의 성공은 새 SD 생성 성공과 구별합니다.


In [ ]:
from luna_genai.sampling_app import build_sampling_app,sampling_record_view
view=sampling_record_view('base')
app=build_sampling_app()
if IN_COLAB:
    app.queue().launch(share=True,inline=True,prevent_thread_lock=True,show_error=True)


In [ ]:
assert len(record['runs'])==4 and record['parity']['passed']
assert np.isfinite(tensors['base_final']).all()
assert not np.array_equal(tensors['base_frame_0'],tensors['base_final'])
assert len(trace)==20 and trace['input_t'].is_monotonic_decreasing
assert view[0].size==(512,512) and len(view[1])==5
assert '지금 SD를 생성하지 않음' in view[2]['mode']
assert Path(view[3]).is_file()
assert len(app.config['dependencies'])>=4
print('W07A A 검증 통과 · 수계산/실제 기록 분석/앱 callback · 새 SD 생성 아님')
if not IN_COLAB: app.close()
